In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.impute import KNNImputer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

ICU = pd.read_csv('Preprocessed ICU data.csv')
print(ICU.shape)

(1474, 232)


# Missing values

In [2]:
missing_counts = ICU.isnull().sum()

print(missing_counts[missing_counts > 0] if missing_counts.sum() > 0 else "No missing values found")

No missing values found


# Duplicated row and column

In [3]:
# Check which columns are duplicates of an earlier column
duplicate_column_mask = ICU.T.duplicated()

# Get the names of the duplicate columns
duplicate_columns = ICU.columns[duplicate_column_mask].tolist()
print("Identical columns found:", duplicate_columns)

# Map each duplicate column to its original match
for col in duplicate_columns:
    for original in ICU.columns:
        # Find the first column that matches it (and isn't itself)
        if original != col and ICU[col].equals(ICU[original]):
            print(f"'{col}' is identical to '{original}'")
            break

Identical columns found: ['Min_MechVent.x', 'Max_MechVent.x', 'Mean_MechVent.y', 'Min_MechVent.y', 'Max_MechVent.y']
'Min_MechVent.x' is identical to 'Mean_MechVent.x'
'Max_MechVent.x' is identical to 'Mean_MechVent.x'
'Mean_MechVent.y' is identical to 'Mean_MechVent.x'
'Min_MechVent.y' is identical to 'Mean_MechVent.x'
'Max_MechVent.y' is identical to 'Mean_MechVent.x'


In [4]:
ICU = ICU.drop_duplicates()
print(ICU.shape)

(1474, 232)


# Identify odd values

In [5]:
print(ICU.dtypes)

RecordID               int64
Age                    int64
Gender               float64
Height               float64
ICUType2               int64
                      ...   
Max_TroponinI.y      float64
Max_TroponinT.y      float64
SAPS.I                 int64
SOFA                   int64
In.hospital_death      int64
Length: 232, dtype: object


### Gender

In [6]:
print(ICU['Gender'].unique())

[1.         0.         0.56192144]


We found an invalid value (0.56192144) in the Gender column, which likely came from a mean imputation. 

We removed these rows using .isin() to keep only the valid binary values (0 and 1).

In [7]:
ICU = ICU[ICU['Gender'].isin([1,0])].copy()
print(ICU['Gender'].unique())

[1. 0.]


### SOFA & SAPS.I

In [8]:
ICU[ICU['SOFA'] == -1]

,RecordID,Age,Gender,Height,ICUType2,ICUType3,ICUType4,Mean_Weight.x,Mean_GCS.x,Mean_HR.x,...,Max_ALT.y,Max_AST.y,Max_Bilirubin.y,Max_Lactate.y,Max_Cholesterol.y,Max_TroponinI.y,Max_TroponinT.y,SAPS.I,SOFA,In.hospital_death
16,132610,72,1.0,172.900000,0,1,0,72.260000,15.000000,74.346154,...,189.169186,260.323666,1.979557,2.395461,156.842105,7.132683,1.114629,-1,-1,0
58,132929,71,0.0,162.600000,0,1,0,113.800000,15.000000,110.956522,...,14.000000,46.000000,2.500000,11.500000,156.842105,7.132683,0.040000,13,-1,1
113,133399,50,1.0,175.300000,0,0,1,114.406818,5.125000,79.862069,...,26.000000,18.000000,1.900000,1.900000,156.842105,7.132683,1.114629,19,-1,1
117,133430,40,1.0,167.600000,0,1,0,199.744444,13.400000,81.185185,...,189.169186,260.323666,1.979557,1.400000,156.842105,7.132683,1.114629,9,-1,0
128,133485,73,0.0,167.600000,0,0,1,66.493548,11.818182,120.130435,...,189.169186,260.323666,1.979557,0.800000,216.000000,7.132683,1.114629,16,-1,0
152,133628,88,1.0,169.787227,0,1,0,83.054136,11.117704,96.000000,...,189.169186,260.323666,1.979557,2.395461,156.842105,7.132683,1.114629,-1,-1,0
185,133834,90,0.0,169.787227,0,1,0,56.500000,12.400000,103.607143,...,189.169186,260.323666,1.979557,2.395461,156.842105,7.132683,1.114629,-1,-1,0
259,134404,90,0.0,154.900000,0,1,0,68.500000,10.857143,99.826087,...,27.000000,23.000000,2.400000,1.900000,156.842105,7.132683,0.070000,19,-1,0
309,134663,81,0.0,169.787227,0,1,0,60.000000,6.111111,74.375000,...,189.169186,260.323666,1.979557,2.395461,156.842105,0.800000,1.114629,21,-1,0
312,134684,79,0.0,154.900000,0,0,0,75.000000,15.000000,83.794118,...,17.000000,24.000000,0.300000,0.700000,116.000000,7.132683,0.230000,14,-1,0


In [9]:
minus_one_counts = (ICU == -1).sum()

# Filter to show only columns that have at least one -1
columns_with_minus_one = minus_one_counts[minus_one_counts > 0]

if not columns_with_minus_one.empty:
    print("Columns containing -1: ")
    print(columns_with_minus_one)
else:
    print("No columns contain the value -1.")


Columns containing -1: 
SAPS.I    71
SOFA      39
dtype: int64


In [10]:
print(ICU[['SAPS.I', 'SOFA']].head(20))

    SAPS.I  SOFA
0        7     1
1       17     2
2       14    11
3       19     8
4       11     0
5       12     3
6       18     6
7        9     2
8       28    16
9       -1     7
10      17    11
11      10     2
12      -1     5
13      22     5
14      13     1
15      21     9
16      -1    -1
17       8     3
18      20    10
19       8     5


In [11]:
ICU['SAPS.I'] = ICU['SAPS.I'].replace(-1, np.nan)
ICU['SOFA'] = ICU['SOFA'].replace(-1, np.nan)

features = ['Age', 'SAPS.I', 'SOFA']
X = ICU[features].copy()

Xm = X.mean()
Xs = X.std()

X_scaled = (X - Xm) / Xs

imputer = KNNImputer(n_neighbors=5)
Xt_scaled = imputer.fit_transform(X_scaled)

Xt_original = (Xt_scaled * Xs.values) + Xm.values

ICU[features] = Xt_original

ICU['SAPS.I'] = ICU['SAPS.I'].round().astype(int)
ICU['SOFA'] = ICU['SOFA'].round().astype(int)

In [12]:
# Make sure there is no more -1 

ICU[ICU['SOFA'] == -1]
ICU[ICU['SAPS.I'] == -1]

,RecordID,Age,Gender,Height,ICUType2,ICUType3,ICUType4,Mean_Weight.x,Mean_GCS.x,Mean_HR.x,...,Max_ALT.y,Max_AST.y,Max_Bilirubin.y,Max_Lactate.y,Max_Cholesterol.y,Max_TroponinI.y,Max_TroponinT.y,SAPS.I,SOFA,In.hospital_death


In [13]:
print(ICU[['SAPS.I', 'SOFA']].head(20))

    SAPS.I  SOFA
0        7     1
1       17     2
2       14    11
3       19     8
4       11     0
5       12     3
6       18     6
7        9     2
8       28    16
9       15     7
10      17    11
11      10     2
12      15     5
13      22     5
14      13     1
15      21     9
16      18     6
17       8     3
18      20    10
19       8     5


### Height

In [14]:
# Check unusually low Height values
ICU[ICU['Height'] < 54.6]

,RecordID,Age,Gender,Height,ICUType2,ICUType3,ICUType4,Mean_Weight.x,Mean_GCS.x,Mean_HR.x,...,Max_ALT.y,Max_AST.y,Max_Bilirubin.y,Max_Lactate.y,Max_Cholesterol.y,Max_TroponinI.y,Max_TroponinT.y,SAPS.I,SOFA,In.hospital_death
660,136967,88.0,0.0,13.7,0,0,1,99.3,15.000000,97.454545,...,189.169186,260.323666,1.200000,1.2,156.842105,7.132683,1.114629,13,3,0
950,138870,58.0,1.0,15.2,0,0,1,124.0,8.857143,87.875000,...,189.169186,260.323666,1.979557,1.2,156.842105,7.132683,1.114629,14,6,0


We identified two unusually low outliers in the height column (13.7 cm and 15.2 cm). 

Assuming these are typo where a decimal point was misplaced, we corrected them by shifting the decimal one place to the right.

13.7 is likely meant to be 137.0 cm and 15.2 is likely meant to be 152.0 cm

In [15]:
ICU.loc[ICU['Height'] == 13.7, 'Height'] = 137.0
ICU.loc[ICU['Height'] == 15.2, 'Height'] = 152.0

In [16]:
# Check strange Height values
ICU[ICU['Height'] > 272]

,RecordID,Age,Gender,Height,ICUType2,ICUType3,ICUType4,Mean_Weight.x,Mean_GCS.x,Mean_HR.x,...,Max_ALT.y,Max_AST.y,Max_Bilirubin.y,Max_Lactate.y,Max_Cholesterol.y,Max_TroponinI.y,Max_TroponinT.y,SAPS.I,SOFA,In.hospital_death
165,133693,87.0,0.0,365.8,0,0,0,83.054136,14.0,59.086957,...,189.169186,260.323666,0.7,2.395461,156.842105,7.132683,1.114629,12,2,0
1173,140479,50.0,0.0,419.1,0,0,0,67.600000,9.2,80.263158,...,111.000000,168.000000,0.8,3.400000,205.000000,7.132683,1.114629,18,9,0


Conversely, the extreme high values (>272 cm) cannot be reconstructed as a simple typo. 

Therefore, we will convert them to NaN and handle them using KNN imputer.

In [18]:
ICU.loc[ICU['Height'] > 272, 'Height'] = np.nan

features_2 = ['Age', 'Height']
X_height = ICU[features_2].copy()

Xm_height = X_height.mean()
Xs_height = X_height.std()

X_scaled_height = (X_height - Xm_height) / Xs_height

imputer_height = KNNImputer(n_neighbors=5)
Xt_scaled_height = imputer_height.fit_transform(X_scaled_height)

Xt_original_height = (Xt_scaled_height * Xs_height.values) + Xm_height.values

ICU['Height'] = Xt_original_height[:, 1] 

In [19]:
ICU[(ICU['Height'] < 100) | (ICU['Height'] > 250)]

,RecordID,Age,Gender,Height,ICUType2,ICUType3,ICUType4,Mean_Weight.x,Mean_GCS.x,Mean_HR.x,...,Max_ALT.y,Max_AST.y,Max_Bilirubin.y,Max_Lactate.y,Max_Cholesterol.y,Max_TroponinI.y,Max_TroponinT.y,SAPS.I,SOFA,In.hospital_death


In [20]:
print(ICU.loc[[165, 1173], ['Age', 'Height']])

       Age  Height
165   87.0  159.02
1173  50.0  174.28
